In [5]:
import json

# 1. 샘플 데이터 로드
mock_data = {
  "name": "project-alpha",
  "description": "None", 
  "language": "Python",
  "topics": [],
  "selectedFileContents": [
    {
      "path": "requirements.txt",
      "content": "pandas==2.0.0\nyfinance==0.2.18\nschedule==1.2.0\nrequests==2.28.2"
    },
    {
      "path": "main.py",
      "content": "import schedule\nimport time\nfrom src.collector import fetch_data\nfrom src.notifier import send_msg\n\ndef job():\n    price = fetch_data('AAPL')\n    if price > 150:\n        send_msg(f'Apple target hit: {price}')\n\nschedule.every().hour.do(job)\nwhile True:\n    schedule.run_pending()\n    time.sleep(1)"
    },
    {
      "path": "src/collector.py",
      "content": "import yfinance as yf\n\ndef fetch_data(ticker):\n    data = yf.Ticker(ticker)\n    return data.history(period='1d')['Close'].iloc[-1]"
    },
    {
      "path": "src/notifier.py",
      "content": "import requests\n\ndef send_msg(text):\n    url = f'https://api.telegram.org/bot{TOKEN}/sendMessage'\n    requests.post(url, data={'chat_id': ID, 'text': text})"
    }
  ]
}

In [43]:
from huggingface_hub import InferenceClient
import json
import time
import os
from dotenv import load_dotenv

# 1. 환경 설정 및 인증 정보 로드
# .env 파일에 저장된 HF_TOKEN(Hugging Face API 키)을 불러옴
load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

# 2. 테스트 대상 모델 리스트 설정
# 분석에 사용할 모델의 별칭(Key)과 실제 Hugging Face 모델 ID(Value)를 매핑
MODELS = {
    "Qwen2.5-Coder-7B": "Qwen/Qwen2.5-Coder-7B-Instruct",
    "Llama-3.1-8B": "meta-llama/Llama-3.1-8B-Instruct"
}

def call_hf_api(model_id, prompt):
    """
    Hugging Face Inference API를 호출하여 모델의 응답을 반환합니다.
    :param model_id: 호출할 모델의 ID
    :param prompt: 모델에게 전달할 프롬프트(질문)
    :return: 모델의 생성 텍스트 또는 에러 메시지
    """
    # Hugging Face 추론 클라이언트 초기화
    client = InferenceClient(model=model_id, token=HF_TOKEN)
    
    try:
        # Chat Completion API 호출 (대화형 인터페이스)
        response = client.chat_completion(
            messages=[
                {
                    "role": "system", 
                    "content": "당신은 코드 분석 전문가입니다. 답변은 반드시 한국어로 작성하세요."
                },
                {
                    "role": "user", 
                    "content": prompt
                }
            ],
            max_tokens=500, # 최대 생성 토큰 수 제한
        )
        # 생성된 응답 내용만 추출하여 반환
        return response.choices[0].message.content
        
    except Exception as e:
        # API 호출 실패 시 에러 내용 반환
        return f"🚨 에러 발생: {str(e)}"

def test_model_inference(model_alias, model_id, data):
    """
    특정 모델에 대해 주어진 데이터를 바탕으로 프롬프트를 구성하고 성능을 테스트합니다.
    :param model_alias: 출력용 모델 이름
    :param model_id: 실제 모델 ID
    :param data: 분석할 소스 데이터 (JSON 형태)
    """
    print(f"\n" + "="*50)
    print(f"🚀 모델 테스트 중: {model_alias} ({model_id})")
    print("="*50)
    
    # 전달받은 데이터를 JSON 문자열로 변환
    context = json.dumps(data, indent=2, ensure_ascii=False)
    
    # 프롬프트 엔지니어링
    prompt = f"""당신은 코드 분석 전문가입니다. 다음은 특정 GitHub 저장소에서 추출한 정보와 코드의 일부다.
이 정보를 분석해서 다음 두 질문에 답하시오. 답변은 반드시 한국어로 작성하시오.

[데이터]
{context}

질문 1: 이 프로젝트는 무엇을 하는 프로젝트인가요? (한 문장으로)
질문 2: 이 프로젝트의 핵심 기능 3가지는 무엇인가요? (리스트 형식으로)
답변:"""
    
    # 추론 시간 측정을 위한 시작 시간 기록
    start_time = time.time()
    result = call_hf_api(model_id, prompt)
    end_time = time.time()
    
    # 결과 및 소요 시간 출력
    print(result)
    print(f"\n⏱️ 소요 시간: {end_time - start_time:.2f}초")
    return result

# 3. 메인 실행 루프
# 정의된 모델 리스트를 순회하며 테스트를 수행
for alias, m_id in MODELS.items():
    test_model_inference(alias, m_id, mock_data)


🚀 모델 테스트 중: Qwen2.5-Coder-7B (Qwen/Qwen2.5-Coder-7B-Instruct)
질문 1: 이 프로젝트는 실시간 주가 데이터를 수집하여 지정된 가격 이상이 될 때 텔레그램으로 알림을 보내는 Python 스크립트입니다.

질문 2:
- 실시간 주가 데이터를 수집하는 `fetch_data` 함수
- 해당 주가가 목표 가격 이상인지 확인하는 로직
- 알림 메시지를 Telegram으로 보내는 `send_msg` 함수

⏱️ 소요 시간: 2.52초

🚀 모델 테스트 중: Llama-3.1-8B (meta-llama/Llama-3.1-8B-Instruct)
질문 1: 이 프로젝트는 일정 시간마다 apple의 현재 주식 가격을 fetching하고, 그 가격이 150원以上인 경우 telegram으로 메시지를 보내는 자동화 프로젝트입니다.

질문 2: 이 프로젝트의 핵심 기능 세 가지는 다음과 같습니다.

1. 주식 가격 fetching: 주식 가격을 웹에서 fetching하는 기능을 담당하는 collector.py의 fetch_data함수가 있습니다.
2. telegram 메시지 sending: 메시지를 telegram으로 보내는 기능을 담당하는 notifier.py의 send_msg함수가 있습니다.
3. 스케줄링: 일정 시간마다 실행되는 기능을 담당하는 schedule 라이브러리의 every().hour도(job)이 있습니다.

⏱️ 소요 시간: 0.39초
